# 102 - Coordinates, Grids, and Units

In 101, you saw that you build grids and pass them to functions.
In this notebook, we will master the grids:

- Build cartesian grids with `make_xy_grid` and understand their centering
- Convert to and from polar coordinates
- Use the separable (1D) form of a grid for speed
- Rasterize apertures with the `prysm.geometry` functions
- Pin down the unit conventions

Grids are used in essentially the entire library, so boring as they
may be we will nail them down first:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from prysm.coordinates import (
    make_xy_grid,
    cart_to_polar,
    polar_to_cart,
    optimize_xy_separable,
)
from prysm import geometry

## `make_xy_grid`

This is the core function, with API signature:

`make_xy_grid(shape, *, dx=0, diameter=0, grid=True)`

`*` in a list of python args means what is to the right
must be done by keyword only.

- pass `diameter` to fix the total width (spacing is `diameter / max(shape)`), or
- pass `dx` to fix the inter-sample spacing directly.

`shape` is a single int (square) or a `(rows, cols)` tuple in numpy axis order.

In [ ]:
x, y = make_xy_grid(128, diameter=10.0)   # 10 mm wide, 128 samples
print('shape   ', x.shape)
print('dx      ', float(x[0, 1] - x[0, 0]))
print('extent  ', float(x.min()), 'to', float(x.max()))

### Centering

prysm grids are **FFT-aligned**: the coordinate axis is
`arange(-(n//2), -(n//2)+n) * dx`, which always contains an exact `0`.  This
holds for both even and odd sample counts.  An even grid is asymmetric about
zero (one more sample on the negative side), an odd grid is symmetric.

You can use non-centered grids.  Generally the only place you will run into
trouble is that the propagation API is written around the grid being this way,
so you will get unintended errors in the form of phase ramps from the offset
not matching if you deviate.

In [ ]:
xe, _ = make_xy_grid(4, dx=1)   # even
xo, _ = make_xy_grid(5, dx=1)   # odd
print('even axis:', xe[0])
print('odd  axis:', xo[0])
print('both contain 0:', (xe == 0).any(), (xo == 0).any())

## Polar coordinates

`cart_to_polar` and `polar_to_cart` convert between representations.  Polar
coordinates are what many polynomial bases (Zernike, Q) want, and what
radial apertures are easiest to express in.

In [ ]:
r, t = cart_to_polar(x, y)
xx, yy = polar_to_cart(r, t)
print('round trips:', np.allclose(xx, x), np.allclose(yy, y))

extent = [x.min(), x.max(), y.min(), y.max()]

fig, axs = plt.subplots(ncols=2, figsize=(9, 4))
im = axs[0].imshow(r, extent=extent)
fig.colorbar(im, ax=axs[0])
axs[0].set(title='r')

im = axs[1].imshow(t, extent=extent, cmap='twilight')
fig.colorbar(im, ax=axs[1])
axs[1].set(title='t');

`cart_to_polar` has a `vec_to_grid` convenience: hand it two 1D vectors and it
broadcasts them into a 2D `(rho, theta)` grid for you.  That is occasionally
handy, but most of the time you pass it a full 2D `x, y`.

## Separable grids: 2N instead of 2N^2

Many calculations are *separable* in `x` and `y` - any radial quantity in `r^2`
instead of r, or many natural x-y separable functions like a rectangle.  For a
separable operation you do not need the full `N x N` meshgrid; two 1D vectors
broadcast together give the same result in `2N` work instead of `2N^2`.

`make_xy_grid(..., grid=False)` returns the 1D vectors, and `optimize_xy_separable`
takes a full 2D grid and hands back the 1D row/column it was built from.  This
is mostly used internally in the library, but here we'll make a circle relying
on the fact that we can transform `r<v` into `r^2 < v^2` as an example:

In [ ]:
x2, y2 = make_xy_grid(128, diameter=10.0)
r2, _ = cart_to_polar(x2, y2)

x, y = optimize_xy_separable(x2, y2)

rsq = x**2 + y**2

circle_2d = geometry.circle(4, r2)

circle_sep = rsq <= (4**2)

fig, axs = plt.subplots(ncols=2, figsize=(8,4))
axs[0].imshow(circle_2d)
axs[0].set(title='2D grid computation')

axs[1].imshow(circle_sep)
axs[1].set(title='2, 1D vector comps')

## Geometry: rasterizing apertures

`prysm.geometry` turns coordinates into masks - the transmission functions of
apertures.  The radial shapes take a radius and the polar `r`; the polygonal
shapes take `x, y`.  A few of the most common:

- `circle(radius, r)` - hard-edged disk; `truecircle` is anti-aliased
- `annulus(rin, rout, r)` - a ring (e.g. a centrally obscured pupil)
- `rectangle(width, x, y, height=None, angle=0)` and `square(x, y)`
- `regular_polygon(sides, radius, x, y, ...)` - hexagons, etc.
- `spider(vanes, width, x, y, ...)` - secondary-support struts

They return float arrays you multiply together to compose a pupil.

In [ ]:
x, y = make_xy_grid(256, diameter=2.0)
r, t = cart_to_polar(x, y)

disk = geometry.circle(0.9, r)
ring = geometry.annulus(0.3, 0.9, r)
hexa = geometry.regular_polygon(6, 0.9, x, y)

outer = geometry.circle(0.9, r)
inner = (1 - geometry.circle(0.25, r))
spid = geometry.spider(3, 0.03, x, y)
obscured = outer * inner * spid 

fig, axs = plt.subplots(ncols=4, figsize=(14, 3.5))
for ax, dat, title in zip(axs, (disk, ring, hexa, obscured),
                          ('circle', 'annulus', 'hexagon', 'obscured + spider')):
    ax.imshow(dat, extent=[-1, 1, -1, 1], cmap='gray')
    ax.set(title=title, xticks=[], yticks=[])

Because these are just arrays over coordinates, anything you can write as a
function of `x, y` or `r, t` is a valid aperture - your own shapes drop in
exactly like the built-ins.